In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# NB_Silver_Customers
# Bronze -> Silver
# ============================================================

SOURCE_TABLE = "bronze_customers"
SILVER_TABLE = "silver_customers"
QUARANTINE_TABLE = "quarantine_customers"

# ------------------------------------------------------------
# 1. Read Bronze
# ------------------------------------------------------------

df = spark.table(SOURCE_TABLE)

# ------------------------------------------------------------
# 2. Standardize data types and strings
# ------------------------------------------------------------

df = (
    df
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("first_name", F.trim(F.col("first_name").cast("string")))
    .withColumn("last_name", F.trim(F.col("last_name").cast("string")))
    .withColumn("email", F.lower(F.trim(F.col("email").cast("string"))))
    .withColumn("country", F.trim(F.col("country").cast("string")))
    .withColumn("signup_date", F.to_date("signup_date"))
)

# ------------------------------------------------------------
# 3. Validation rules
# ------------------------------------------------------------

email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

df_checked = (
    df
    .withColumn(
        "dq_reason",
        F.when(
            F.col("customer_id").isNull(),
            F.lit("NULL_CUSTOMER_ID")
        )
        .when(
            F.col("email").isNull() |
            (F.col("email") == ""),
            F.lit("NULL_OR_EMPTY_EMAIL")
        )
        .when(
            ~F.col("email").rlike(email_pattern),
            F.lit("INVALID_EMAIL")
        )
        .when(
            F.col("signup_date").isNull(),
            F.lit("NULL_SIGNUP_DATE")
        )
        .when(
            F.col("signup_date") > F.current_date(),
            F.lit("FUTURE_SIGNUP_DATE")
        )
        .otherwise(F.lit(None))
    )
)

# ------------------------------------------------------------
# 4. Quarantine invalid records
# ------------------------------------------------------------

df_quarantine = (
    df_checked
    .filter(F.col("dq_reason").isNotNull())
    .withColumn("quarantined_at", F.current_timestamp())
)

df_quarantine.write.mode("overwrite").format("delta").saveAsTable(
    QUARANTINE_TABLE
)

# ------------------------------------------------------------
# 5. Keep valid records
# ------------------------------------------------------------

df_valid = (
    df_checked
    .filter(F.col("dq_reason").isNull())
    .drop("dq_reason")
)

# ------------------------------------------------------------
# 6. Deduplicate customer_id
# ------------------------------------------------------------

window = Window.partitionBy("customer_id").orderBy(
    F.col("signup_date").desc_nulls_last()
)

df_dedup = (
    df_valid
    .withColumn("row_number", F.row_number().over(window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

# ------------------------------------------------------------
# 7. Write Silver
# ------------------------------------------------------------

df_dedup.write.mode("overwrite").format("delta").saveAsTable(
    SILVER_TABLE
)

print(f"Silver table created: {SILVER_TABLE}")
print(f"Valid records: {df_dedup.count()}")
print(f"Quarantined records: {df_quarantine.count()}")

StatementMeta(, cff7a6fa-8ec1-4fb1-9d94-95b08439e6f6, 3, Finished, Available, Finished, False)

Silver table created: silver_customers
Valid records: 9996
Quarantined records: 3
